# 02 — Broadcast vs Shuffle Joins

Broadcast join and shuffle join behavior in Spark.

## 00 — Setup

```text
SparkSession
    │
    ├── warehouse: /tmp/spark-warehouse
    ├── shuffle partitions: 8
    └── AQE disabled for deterministic physical plans
```

In [1]:
from pyspark.sql import SparkSession, functions as F
import time

spark = (
    SparkSession.builder
    .appName("broadcast_vs_shuffle")
    .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse")
    .getOrCreate()
)

spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.sql.adaptive.enabled", "false")

spark

## 01 — Sample data

```text
fact_sales
  customer_id ─────────────┐
  product_id               │
  quantity                 │
  price                    │
                            ├── join key
customers                  │
  customer_id ─────────────┘
  segment
  country

small_segments
  segment ──────────────── customer segment enrichment
```

In [2]:
fact_sales = (
    spark.range(0, 1_000_000)
    .withColumnRenamed("id", "sale_id")
    .withColumn("customer_id", (F.col("sale_id") % 100_000).cast("long"))
    .withColumn("product_id", (F.col("sale_id") % 10_000).cast("long"))
    .withColumn("quantity", ((F.col("sale_id") % 5) + 1).cast("int"))
    .withColumn("price", ((F.col("sale_id") % 200) + 20).cast("double"))
)

customers = (
    spark.range(0, 100_000)
    .withColumnRenamed("id", "customer_id")
    .withColumn("segment", F.expr("CASE WHEN customer_id % 3 = 0 THEN 'enterprise' WHEN customer_id % 3 = 1 THEN 'smb' ELSE 'consumer' END"))
    .withColumn("country", F.expr("CASE WHEN customer_id % 4 = 0 THEN 'SK' WHEN customer_id % 4 = 1 THEN 'CZ' WHEN customer_id % 4 = 2 THEN 'DE' ELSE 'AT' END"))
)

small_segments = spark.createDataFrame(
    [
        ("enterprise", "high_touch"),
        ("smb", "standard"),
        ("consumer", "self_service"),
    ],
    ["segment", "support_model"]
)

fact_sales.count(), customers.count(), small_segments.count()

(1000000, 100000, 3)

## 02 — Shuffle join

```text
fact_sales partitions                 customers partitions
        │                                      │
        ├──── shuffle by customer_id ──────────┤
        │                                      │
        ▼                                      ▼
  same customer_id values meet in the same reducer
        │
        ▼
  SortMergeJoin / ShuffledHashJoin
```

In [3]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

shuffle_join = fact_sales.join(customers, on="customer_id", how="inner")
shuffle_join.explain(mode="simple")

== Physical Plan ==
*(5) Project [customer_id#2L, sale_id#1L, product_id#3L, quantity#4, price#5, segment#8, country#9]
+- *(5) SortMergeJoin [customer_id#2L], [customer_id#7L], Inner
   :- *(2) Sort [customer_id#2L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(customer_id#2L, 8), ENSURE_REQUIREMENTS, [plan_id=117]
   :     +- *(1) Project [id#0L AS sale_id#1L, (id#0L % 100000) AS customer_id#2L, (id#0L % 10000) AS product_id#3L, cast(((id#0L % 5) + 1) as int) AS quantity#4, cast(((id#0L % 200) + 20) as double) AS price#5]
   :        +- *(1) Filter isnotnull((id#0L % 100000))
   :           +- *(1) Range (0, 1000000, step=1, splits=2)
   +- *(4) Sort [customer_id#7L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(customer_id#7L, 8), ENSURE_REQUIREMENTS, [plan_id=123]
         +- *(3) Project [id#6L AS customer_id#7L, CASE WHEN ((id#6L % 3) = 0) THEN enterprise WHEN ((id#6L % 3) = 1) THEN smb ELSE consumer END AS segment#8, CASE WHEN ((id#6L % 4) = 0) THEN

## 03 — Broadcast join

```text
customers
    │
    ├── collect + broadcast
    ▼
executor 1  executor 2  executor 3  executor N
    │           │           │           │
    └──── local hash lookup against fact partitions

fact_sales does not shuffle
```

In [4]:
broadcast_join = fact_sales.join(F.broadcast(customers), on="customer_id", how="inner")
broadcast_join.explain(mode="simple")

== Physical Plan ==
*(2) Project [customer_id#2L, sale_id#1L, product_id#3L, quantity#4, price#5, segment#8, country#9]
+- *(2) BroadcastHashJoin [customer_id#2L], [customer_id#7L], Inner, BuildRight, false
   :- *(2) Project [id#0L AS sale_id#1L, (id#0L % 100000) AS customer_id#2L, (id#0L % 10000) AS product_id#3L, cast(((id#0L % 5) + 1) as int) AS quantity#4, cast(((id#0L % 200) + 20) as double) AS price#5]
   :  +- *(2) Filter isnotnull((id#0L % 100000))
   :     +- *(2) Range (0, 1000000, step=1, splits=2)
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=176]
      +- *(1) Project [id#6L AS customer_id#7L, CASE WHEN ((id#6L % 3) = 0) THEN enterprise WHEN ((id#6L % 3) = 1) THEN smb ELSE consumer END AS segment#8, CASE WHEN ((id#6L % 4) = 0) THEN SK WHEN ((id#6L % 4) = 1) THEN CZ WHEN ((id#6L % 4) = 2) THEN DE ELSE AT END AS country#9]
         +- *(1) Range (0, 100000, step=1, splits=2)




## 04 — Tiny dimension broadcast

```text
customers ── join(segment) ── small_segments
                              │
                              └── ideal broadcast side
```

In [5]:
tiny_broadcast = customers.join(F.broadcast(small_segments), on="segment", how="left")
tiny_broadcast.explain(mode="simple")
tiny_broadcast.groupBy("support_model").count().orderBy("support_model").show()

== Physical Plan ==
*(2) Project [segment#8, customer_id#7L, country#9, support_model#11]
+- *(2) BroadcastHashJoin [segment#8], [segment#10], LeftOuter, BuildRight, false
   :- *(2) Project [id#6L AS customer_id#7L, CASE WHEN ((id#6L % 3) = 0) THEN enterprise WHEN ((id#6L % 3) = 1) THEN smb ELSE consumer END AS segment#8, CASE WHEN ((id#6L % 4) = 0) THEN SK WHEN ((id#6L % 4) = 1) THEN CZ WHEN ((id#6L % 4) = 2) THEN DE ELSE AT END AS country#9]
   :  +- *(2) Range (0, 100000, step=1, splits=2)
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=213]
      +- *(1) Filter isnotnull(segment#10)
         +- *(1) Scan ExistingRDD[segment#10,support_model#11]


+-------------+-----+
|support_model|count|
+-------------+-----+
|   high_touch|33334|
| self_service|33333|
|     standard|33333|
+-------------+-----+



## 05 — Auto broadcast threshold

```text
Spark checks estimated size
        │
        ├── below threshold  → BroadcastHashJoin
        └── above threshold  → shuffle-based join
```

In [6]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

auto_join = customers.join(small_segments, on="segment", how="left")
auto_join.explain(mode="simple")

== Physical Plan ==
*(5) Project [segment#8, customer_id#7L, country#9, support_model#11]
+- *(5) SortMergeJoin [segment#8], [segment#10], LeftOuter
   :- *(2) Sort [segment#8 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(segment#8, 8), ENSURE_REQUIREMENTS, [plan_id=327]
   :     +- *(1) Project [id#6L AS customer_id#7L, CASE WHEN ((id#6L % 3) = 0) THEN enterprise WHEN ((id#6L % 3) = 1) THEN smb ELSE consumer END AS segment#8, CASE WHEN ((id#6L % 4) = 0) THEN SK WHEN ((id#6L % 4) = 1) THEN CZ WHEN ((id#6L % 4) = 2) THEN DE ELSE AT END AS country#9]
   :        +- *(1) Range (0, 100000, step=1, splits=2)
   +- *(4) Sort [segment#10 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(segment#10, 8), ENSURE_REQUIREMENTS, [plan_id=333]
         +- *(3) Filter isnotnull(segment#10)
            +- *(3) Scan ExistingRDD[segment#10,support_model#11]




## 06 — Disable broadcast and compare plan

```text
same logical join
        │
        ├── broadcast enabled  → broadcast physical plan
        └── broadcast disabled → shuffle physical plan
```

In [7]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

no_auto_broadcast = customers.join(small_segments, on="segment", how="left")
no_auto_broadcast.explain(mode="simple")

== Physical Plan ==
*(5) Project [segment#8, customer_id#7L, country#9, support_model#11]
+- *(5) SortMergeJoin [segment#8], [segment#10], LeftOuter
   :- *(2) Sort [segment#8 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(segment#8, 8), ENSURE_REQUIREMENTS, [plan_id=385]
   :     +- *(1) Project [id#6L AS customer_id#7L, CASE WHEN ((id#6L % 3) = 0) THEN enterprise WHEN ((id#6L % 3) = 1) THEN smb ELSE consumer END AS segment#8, CASE WHEN ((id#6L % 4) = 0) THEN SK WHEN ((id#6L % 4) = 1) THEN CZ WHEN ((id#6L % 4) = 2) THEN DE ELSE AT END AS country#9]
   :        +- *(1) Range (0, 100000, step=1, splits=2)
   +- *(4) Sort [segment#10 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(segment#10, 8), ENSURE_REQUIREMENTS, [plan_id=391]
         +- *(3) Filter isnotnull(segment#10)
            +- *(3) Scan ExistingRDD[segment#10,support_model#11]




## 07 — Runtime comparison

```text
DataFrame
   │
   ├── action: count()
   └── measured runtime
```

In [8]:
def measure(label, df):
    start = time.time()
    rows = df.count()
    elapsed = time.time() - start
    return (label, rows, round(elapsed, 3))

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

results = [
    measure("shuffle_join", shuffle_join),
    measure("broadcast_join", broadcast_join),
]

spark.createDataFrame(results, ["case", "rows", "seconds"]).show(truncate=False)

+--------------+-------+-------+
|case          |rows   |seconds|
+--------------+-------+-------+
|shuffle_join  |1000000|1.116  |
|broadcast_join|1000000|0.389  |
+--------------+-------+-------+



## 08 — Broadcast is not always better

```text
broadcast side too large
        │
        ├── driver collects relation
        ├── executors store copy
        └── memory pressure / timeout risk
```

In [9]:
large_customers = (
    spark.range(0, 2_000_000)
    .withColumnRenamed("id", "customer_id")
    .withColumn("payload", F.sha2(F.col("customer_id").cast("string"), 256))
)

large_broadcast_candidate = fact_sales.join(F.broadcast(large_customers), on="customer_id", how="inner")
large_broadcast_candidate.explain(mode="simple")

== Physical Plan ==
*(2) Project [customer_id#2L, sale_id#1L, product_id#3L, quantity#4, price#5, payload#86]
+- *(2) BroadcastHashJoin [customer_id#2L], [customer_id#85L], Inner, BuildRight, false
   :- *(2) Project [id#0L AS sale_id#1L, (id#0L % 100000) AS customer_id#2L, (id#0L % 10000) AS product_id#3L, cast(((id#0L % 5) + 1) as int) AS quantity#4, cast(((id#0L % 200) + 20) as double) AS price#5]
   :  +- *(2) Filter isnotnull((id#0L % 100000))
   :     +- *(2) Range (0, 1000000, step=1, splits=2)
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=621]
      +- *(1) Project [id#84L AS customer_id#85L, sha2(cast(cast(id#84L as string) as binary), 256) AS payload#86]
         +- *(1) Range (0, 2000000, step=1, splits=2)




## 09 — Join hints comparison

```text
same DataFrames
    │
    ├── BROADCAST hint
    ├── MERGE hint
    └── SHUFFLE_HASH hint
```

In [10]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

hint_cases = {
    "broadcast_hint": fact_sales.join(F.broadcast(customers), "customer_id"),
    "merge_hint": fact_sales.hint("MERGE").join(customers.hint("MERGE"), "customer_id"),
    "shuffle_hash_hint": fact_sales.hint("SHUFFLE_HASH").join(customers.hint("SHUFFLE_HASH"), "customer_id"),
}

for name, df in hint_cases.items():
    print(f"\n{'=' * 80}\n{name}\n{'=' * 80}")
    df.explain(mode="simple")


broadcast_hint
== Physical Plan ==
*(2) Project [customer_id#2L, sale_id#1L, product_id#3L, quantity#4, price#5, segment#8, country#9]
+- *(2) BroadcastHashJoin [customer_id#2L], [customer_id#7L], Inner, BuildRight, false
   :- *(2) Project [id#0L AS sale_id#1L, (id#0L % 100000) AS customer_id#2L, (id#0L % 10000) AS product_id#3L, cast(((id#0L % 5) + 1) as int) AS quantity#4, cast(((id#0L % 200) + 20) as double) AS price#5]
   :  +- *(2) Filter isnotnull((id#0L % 100000))
   :     +- *(2) Range (0, 1000000, step=1, splits=2)
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=662]
      +- *(1) Project [id#6L AS customer_id#7L, CASE WHEN ((id#6L % 3) = 0) THEN enterprise WHEN ((id#6L % 3) = 1) THEN smb ELSE consumer END AS segment#8, CASE WHEN ((id#6L % 4) = 0) THEN SK WHEN ((id#6L % 4) = 1) THEN CZ WHEN ((id#6L % 4) = 2) THEN DE ELSE AT END AS country#9]
         +- *(1) Range (0, 100000, step=1, splits=2)



merge_hint
== Physical Plan 

## 10 — Decision table

```text
small dimension table
        │
        ├── yes → broadcast
        │
        └── no
             │
             ├── both sides large → shuffle join
             ├── sorted/bucketed data → merge-friendly
             └── skewed key → AQE skew handling or salting
```

In [11]:
decision_rows = [
    ("small dimension", "BroadcastHashJoin", "Avoids large-side shuffle"),
    ("both sides large", "SortMergeJoin", "Stable default for large joins"),
    ("large but hash-friendly", "ShuffledHashJoin", "Can be faster when build side fits per partition"),
    ("skewed keys", "AQE skew split / salting", "Avoids one reducer doing most work"),
    ("already bucketed", "Bucket-aware join", "Can reduce shuffle when layouts match"),
]

spark.createDataFrame(decision_rows, ["case", "preferred_strategy", "reason"]).show(truncate=False)

+-----------------------+------------------------+------------------------------------------------+
|case                   |preferred_strategy      |reason                                          |
+-----------------------+------------------------+------------------------------------------------+
|small dimension        |BroadcastHashJoin       |Avoids large-side shuffle                       |
|both sides large       |SortMergeJoin           |Stable default for large joins                  |
|large but hash-friendly|ShuffledHashJoin        |Can be faster when build side fits per partition|
|skewed keys            |AQE skew split / salting|Avoids one reducer doing most work              |
|already bucketed       |Bucket-aware join       |Can reduce shuffle when layouts match           |
+-----------------------+------------------------+------------------------------------------------+

